# 🏛️ Nano ChessGPT — 85M Parameter Training

**Character-level chess language model trained on 26M Lichess Elite games.**

| Property | Value |
|----------|-------|
| Architecture | GPT-2 Small (12L × 12H × 768D) |
| Parameters | ~85 Million |
| Dataset | Lichess Elite DB (9.19B tokens, 16.27 GB) |
| Vocab | 29 chars (character-level) |
| Target | ~1500-1800 ELO |

---
### 📋 Every Session Workflow (after first time)
1. **Cell 1** — GPU check  
2. **Cell 2** — Mount Drive + copy dataset (~10-15 min)  
3. **Cell 3** — Clone repo  
4. **Cell 4** — Start / Resume Training  
5. **Cell 5** — Save checkpoint to Drive (before session expires!)  

> ✅ Dataset is stored in Google Drive — **no re-downloading needed!**  
> ⚡ After disconnect: Run Cells 1→3, then Cell 4 (auto-resumes)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — GPU & Environment Check
# ════════════════════════════════════════════════════════
import torch, os, subprocess, psutil

print('=' * 55)
print(' Nano ChessGPT 85M — Environment Check')
print('=' * 55)

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    bf16 = torch.cuda.is_bf16_supported()
    print(f'\n✅ GPU  : {gpu}')
    print(f'   VRAM : {vram:.1f} GB')
    print(f'   bf16 : {"✅ supported" if bf16 else "⚠️  not supported"}')
    print(f'   PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
else:
    print('❌ No GPU! → Runtime → Change runtime type → T4 GPU')
    raise SystemExit('Please enable GPU first!')

result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
disk   = result.stdout.strip().split('\n')[1].split()
ram    = psutil.virtual_memory()
print(f'\n💾 Disk: {disk[3]} free of {disk[1]}')
print(f'   RAM : {ram.available/1024**3:.1f} GB free of {ram.total/1024**3:.1f} GB')
print(f'\n✅ Ready!')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive + Copy Dataset
# Dataset in Drive: NanoChessGPT/train.bin (16.27 GB)
# This cell copies it to Colab local disk (~10-15 min)
# ════════════════════════════════════════════════════════
from google.colab import drive
import os, shutil, time

# Mount Drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

# Paths
DRIVE_DATA_DIR    = '/content/drive/MyDrive/NanoChessGPT'
DRIVE_CKPT_DIR    = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
LOCAL_DATA_DIR    = '/content/chess_data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Check files in Drive
required = ['train.bin', 'val.bin', 'meta.pkl']
print(f'\n📁 Checking Drive: {DRIVE_DATA_DIR}')
for f in required:
    path = f'{DRIVE_DATA_DIR}/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024**3
        print(f'  ✅ {f:12s} ({size:.2f} GB)')
    else:
        print(f'  ❌ {f:12s} — NOT FOUND in Drive!')
        print(f'     Upload it to Google Drive first!')

# Copy dataset to local Colab disk
print(f'\n📦 Copying dataset to local disk...')
for f in required:
    src = f'{DRIVE_DATA_DIR}/{f}'
    dst = f'{LOCAL_DATA_DIR}/{f}'
    if os.path.exists(dst):
        print(f'  ✅ {f} already local — skip')
        continue
    if not os.path.exists(src):
        continue
    size_gb = os.path.getsize(src) / 1024**3
    print(f'  ⏳ Copying {f} ({size_gb:.2f} GB)...', end='', flush=True)
    t0 = time.time()
    shutil.copy2(src, dst)
    elapsed = time.time() - t0
    speed   = size_gb / elapsed * 1024  # MB/s
    print(f' done in {elapsed/60:.1f} min ({speed:.0f} MB/s)')

# Check existing checkpoints
ckpts = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pt')]
print(f'\n🔄 Checkpoints in Drive: {len(ckpts)}')
for c in sorted(ckpts):
    size = os.path.getsize(f'{DRIVE_CKPT_DIR}/{c}') / 1024**2
    print(f'   - {c} ({size:.0f} MB)')

print(f'\n✅ Dataset ready at: {LOCAL_DATA_DIR}')
print(f'   Run Cell 3 → Cell 4 to train!')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — Clone Repo & Setup Paths
# ════════════════════════════════════════════════════════
import os, shutil

REPO_URL       = 'https://github.com/ItsMohammadSajid/NanoChessGPT-85M.git'
REPO_DIR       = '/content/NanoChessGPT-85M'
LOCAL_DATA_DIR = '/content/chess_data'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
LOCAL_OUT_DIR  = f'{REPO_DIR}/out-chess-gpt2small'

# Clone repo
if os.path.exists(REPO_DIR):
    print(f'✅ Repo already cloned at {REPO_DIR}')
else:
    print('📦 Cloning NanoChessGPT-85M...')
    !git clone {REPO_URL} {REPO_DIR}
    print('✅ Cloned!')

os.chdir(REPO_DIR)
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

# Symlink dataset into expected location
CHESS_DATA_DIR = f'{REPO_DIR}/data/chess'
os.makedirs(CHESS_DATA_DIR, exist_ok=True)

for f in ['train.bin', 'val.bin', 'meta.pkl']:
    src = f'{LOCAL_DATA_DIR}/{f}'
    dst = f'{CHESS_DATA_DIR}/{f}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'  🔗 Linked {f}')
    elif os.path.exists(dst):
        print(f'  ✅ {f} already linked')
    else:
        print(f'  ❌ {f} missing from {LOCAL_DATA_DIR}! Run Cell 2 first.')

# Copy Drive checkpoints to local out dir
if os.path.exists(DRIVE_CKPT_DIR):
    ckpts = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pt')]
    if ckpts:
        print(f'\n🔄 Copying {len(ckpts)} checkpoint(s) from Drive...')
        for c in ckpts:
            shutil.copy2(f'{DRIVE_CKPT_DIR}/{c}', f'{LOCAL_OUT_DIR}/{c}')
            print(f'   ✅ {c}')
    else:
        print(f'\n🆕 No checkpoints — fresh training')

print(f'\n✅ Setup complete!')
print(f'   Working dir : {os.getcwd()}')
print(f'   Dataset     : {CHESS_DATA_DIR}')
print(f'   Checkpoints : {LOCAL_OUT_DIR}')
!ls -lh data/chess/

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — Start / Resume Training
# Auto-detects whether to start fresh or resume
# ════════════════════════════════════════════════════════
import os

REPO_DIR      = '/content/NanoChessGPT-85M'
LOCAL_OUT_DIR = f'{REPO_DIR}/out-chess-gpt2small'
TRAIN_BIN     = f'{REPO_DIR}/data/chess/train.bin'

os.chdir(REPO_DIR)

# Verify dataset
if not os.path.exists(TRAIN_BIN):
    raise FileNotFoundError('❌ train.bin not found! Run Cells 2 & 3 first.')
size_gb = os.path.getsize(TRAIN_BIN) / 1024**3
print(f'✅ Dataset: train.bin ({size_gb:.1f} GB)')

# Detect start vs resume
existing = [f for f in os.listdir(LOCAL_OUT_DIR) if f.endswith('.pt')] if os.path.exists(LOCAL_OUT_DIR) else []
INIT_FROM = 'resume' if existing else 'scratch'
print(f'   Mode: {INIT_FROM.upper()} {"(" + sorted(existing)[-1] + ")" if existing else ""}')

print(f'\n🚀 Training started...')
print(f'   ⚠️  Run Cell 5 to save checkpoint BEFORE session expires!\n')

!python train.py config/train_chess_gpt2small.py \
    --init_from={INIT_FROM} \
    --out_dir=out-chess-gpt2small

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — 💾 Save Checkpoint to Drive
# ⚠️  Run this BEFORE session expires!
# ════════════════════════════════════════════════════════
import os, shutil, glob

LOCAL_OUT_DIR  = '/content/NanoChessGPT-85M/out-chess-gpt2small'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

ckpt_files = glob.glob(f'{LOCAL_OUT_DIR}/*.pt')

if not ckpt_files:
    print('❌ No checkpoint files found!')
    print(f'   Training must run first (Cell 4)')
else:
    print(f'💾 Saving {len(ckpt_files)} checkpoint(s) to Drive...')
    for f in sorted(ckpt_files):
        fname    = os.path.basename(f)
        dst      = f'{DRIVE_CKPT_DIR}/{fname}'
        size_mb  = os.path.getsize(f) / 1024**2
        print(f'   Copying {fname} ({size_mb:.0f} MB)...', end='', flush=True)
        shutil.copy2(f, dst)
        print(' ✅')

    print(f'\n✅ All saved to: {DRIVE_CKPT_DIR}')
    print(f'\n📋 Next session:')
    print(f'   Run Cells 1 → 2 → 3 → 4')
    print(f'   Training will AUTO-RESUME from checkpoint!')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — 📊 Training Progress Monitor
# ════════════════════════════════════════════════════════
import torch, os, glob

LOCAL_OUT_DIR = '/content/NanoChessGPT-85M/out-chess-gpt2small'
ckpt_files    = glob.glob(f'{LOCAL_OUT_DIR}/*.pt')

if not ckpt_files:
    print('No checkpoints yet. Run Cell 4 to start training.')
else:
    for ckpt_path in sorted(ckpt_files):
        ckpt       = torch.load(ckpt_path, map_location='cpu')
        iter_num   = ckpt.get('iter_num', 0)
        best_val   = ckpt.get('best_val_loss', float('inf'))
        max_iters  = ckpt.get('config', {}).get('max_iters', 600000)
        progress   = iter_num / max_iters * 100
        remaining  = (max_iters - iter_num) * 1.2 / 3600

        print(f'📊 {os.path.basename(ckpt_path)}')
        print(f'   Progress  : {iter_num:,} / {max_iters:,} iters ({progress:.1f}%)')
        print(f'   Val loss  : {best_val:.4f}')
        print(f'   Remaining : ~{remaining:.1f} hours (est. on T4)')
        print(f'   Bar       : [{"█" * int(progress/5)}{"░" * (20-int(progress/5))}] {progress:.0f}%')

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — 🎮 Generate Chess Moves (after training)
# ════════════════════════════════════════════════════════
import os
os.chdir('/content/NanoChessGPT-85M')

START_MOVES = "e4 e5 Nf3 "  # Change this to test different positions

!python sample.py \
    --out_dir=out-chess-gpt2small \
    --start="{START_MOVES}" \
    --num_samples=5 \
    --max_new_tokens=300 \
    --temperature=0.8 \
    --top_k=10 \
    --device=cuda